# RL Experiment 04: Specialist Agents

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `load_context()`, `create_env()` |
| `src/schedule_engine/rl/multi_agent/` | Specialist agent system | `AgentCoordinator` |
| **This notebook** | Experiment-specific config | Agent selection strategy |

## Experiment Overview
- **System**: Multi-agent coordinator with specialist agents
- **Goal**: Demonstrate state-based agent selection
- **Strategies**: State-based selection based on GA search state
- **Metrics**: Agent selection patterns across episodes

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks import (
    build_notebook_config,
    create_env,
    load_context,
    set_global_seed,
)
from schedule_engine.rl.multi_agent.agent_coordinator import AgentCoordinator

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Configuration (Inline - Experiment-Specific)

In [ ]:

# RL EXPERIMENT 04 CONFIGURATION - Specialist Agents


SEED = 42
POP_SIZE = 20
MAX_GENERATIONS = 50
MAX_STEPS = 15
NUM_EPISODES = 5  # Number of episodes to run for selection pattern analysis

# Paths - Organized by experiment with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/notebooks/rl_04_specialist_{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Config: pop={POP_SIZE}, ngen={MAX_GENERATIONS}, episodes={NUM_EPISODES}")
print(f" Output: {OUTPUT_DIR}")

## 3. Load Data & Create Environment

In [ ]:
# Set reproducibility
set_global_seed(SEED)

# Build config and load scheduling context
config = build_notebook_config(seed=SEED, overrides={"pop_size": POP_SIZE})
_, context = load_context(DATA_DIR, config)

# Create RL environment
env = create_env(
    context=context,
    pop_size=POP_SIZE,
    max_generations=MAX_GENERATIONS,
    max_steps=MAX_STEPS,
)

print(f" Environment created: obs_space={env.observation_space.shape}, action_space={env.action_space.n}")

## 4. Run Agent Selection Loop

In [ ]:
# Create specialist agent coordinator
coordinator = AgentCoordinator(strategy="state_based")

# Track agent selection patterns
selection_history = []

for episode in range(NUM_EPISODES):
    obs, info = env.reset()
    episode_selections = []
    
    for step in range(MAX_STEPS):
        # Get current state for agent selection
        state = {
            "generation": info.get("generation", 0),
            "generations_without_improvement": info.get("generations_without_improvement", 0),
        }
        
        # Select specialist agent based on state
        agent = coordinator.select_agent(env.population, state, obs)
        episode_selections.append(agent.name)
        
        # Take action (random for demonstration)
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        
        if terminated or truncated:
            break
    
    selection_history.append(episode_selections)
    print(f"Episode {episode+1}: Selected agents: {episode_selections[:5]}...")  # Show first 5

print(f"\n Completed {NUM_EPISODES} episodes")

## 5. Analyze Selection Patterns

In [ ]:
from collections import Counter

# Aggregate selection statistics
all_selections = [sel for episode in selection_history for sel in episode]
selection_counts = Counter(all_selections)

print(f"\n{'='*50}")
print(f"RL EXPERIMENT 04: SPECIALIST AGENTS RESULTS")
print(f"{'='*50}")
print(f"Total selections: {len(all_selections)}")
print(f"\nAgent selection distribution:")
for agent_name, count in selection_counts.most_common():
    pct = 100 * count / len(all_selections)
    print(f"  {agent_name}: {count} ({pct:.1f}%)")
print(f"{'='*50}")

## 6. Save Results

In [ ]:
import json

# Save experiment results
results_data = {
    "experiment": "rl_04_specialist_agents",
    "timestamp": TIMESTAMP,
    "config": {
        "seed": SEED,
        "pop_size": POP_SIZE,
        "max_generations": MAX_GENERATIONS,
        "max_steps": MAX_STEPS,
        "num_episodes": NUM_EPISODES,
        "strategy": "state_based",
    },
    "results": {
        "total_selections": len(all_selections),
        "selection_distribution": dict(selection_counts),
        "selection_history": selection_history,
    },
}

results_path = OUTPUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results_data, f, indent=2)

print(f" Results saved to: {results_path}")